# Unsupervised Learning: K-Means Clustering & PCA pada Dataset Iris

**Notebook ini merupakan bagian dari portfolio Machine Learning.**

Topik: **Unsupervised Learning** — yaitu pendekatan pembelajaran mesin di mana data yang digunakan **tidak memiliki label**. Model harus menemukan struktur atau pola tersembunyi dalam data secara mandiri.

Dalam notebook ini kita akan menggunakan:
- **K-Means Clustering** — algoritma pengelompokan berbasis jarak yang mempartisi data ke dalam *k* cluster.
- **PCA (Principal Component Analysis)** — teknik reduksi dimensi yang memproyeksikan data ke komponen utama (principal components) untuk mempertahankan variansi maksimal.

Dataset yang digunakan adalah **Iris** (built-in dari `sklearn.datasets`), yang terdiri dari 150 sampel bunga Iris dengan 4 fitur numerik (sepal length, sepal width, petal length, petal width) dan 3 spesies (setosa, versicolor, virginica).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
iris = load_iris()
X = iris.data
y_true = iris.target
target_names = iris.target_names
feature_names = iris.feature_names

df = pd.DataFrame(X, columns=feature_names)
df['species'] = pd.Categorical.from_codes(y_true, target_names)

print(f'Ukuran dataset: {X.shape}')
print(f'Jumlah sampel per spesies:')
print(df['species'].value_counts())
df.head()

## 1. PCA — Reduksi Dimensi

Dataset Iris memiliki 4 fitur. Untuk keperluan visualisasi, kita reduksi dimensi menjadi **2 komponen utama** menggunakan PCA. Sebelum PCA, kita *standardize* fitur agar skala tidak mendominasi komponen.

PCA bekerja dengan mencari arah (vektor) yang memaksimalkan variansi data. Komponen pertama (PC1) menangkap variansi terbesar, diikuti PC2, dan seterusnya.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['species'] = df['species']

print(f'Explained variance ratio: {pca.explained_variance_ratio_}')
print(f'Total variance captured by 2 components: {pca.explained_variance_ratio_.sum():.3f}')

In [ ]:
plt.figure(figsize=(10, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71']

for i, species in enumerate(target_names):
    subset = df_pca[df_pca['species'] == species]
    plt.scatter(subset['PC1'], subset['PC2'],
                c=colors[i], label=species.capitalize(),
                edgecolors='k', alpha=0.7, s=60)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('PCA — Iris Dataset (2 Komponen Utama)')
plt.legend()
plt.tight_layout()
plt.show()

### Analisis Hasil PCA

Dari scatter plot di atas terlihat bahwa:
- Spesies **setosa** (merah) terpisah secara linear dari dua spesies lainnya. Ini menandakan bahwa setosa memiliki karakteristik yang sangat berbeda.
- Spesies **versicolor** (biru) dan **virginica** (hijau) masih agak tumpang tindih, menunjukkan kemiripan fitur di antara keduanya.
- Dua komponen PCA berhasil menangkap sekitar **95%+ variansi** dari data 4 dimensi, artinya proyeksi 2D ini sudah representatif.

## 2. K-Means Clustering (k=3)

Kita tahu dataset Iris memiliki 3 spesies, jadi kita set **k=3**. K-Means bekerja dengan:
1. Inisialisasi *k* centroid secara acak.
2. Assign setiap titik ke centroid terdekat.
3. Perbarui centroid sebagai rata-rata titik dalam cluster.
4. Ulangi langkah 2–3 sampai konvergen.

Kita akan melakukan clustering pada data hasil PCA (2D) agar mudah divisualisasikan.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_pca)

df_pca['cluster'] = cluster_labels
centroids = kmeans.cluster_centers_

In [ ]:
plt.figure(figsize=(10, 6))

for cluster_id in range(3):
    subset = df_pca[df_pca['cluster'] == cluster_id]
    plt.scatter(subset['PC1'], subset['PC2'],
                c=colors[cluster_id], label=f'Cluster {cluster_id}',
                edgecolors='k', alpha=0.7, s=60)

plt.scatter(centroids[:, 0], centroids[:, 1],
            c='black', marker='X', s=200,
            label='Centroids', edgecolors='white', linewidths=1.5)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('K-Means Clustering (k=3) pada Data Iris — Hasil PCA')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Perbandingan Cluster dengan Label Asli

Meskipun K-Means tidak mengetahui label asli, kita bisa membandingkan hasil clusternya dengan spesies sebenarnya. Karena label cluster bersifat arbitrer (angka 0,1,2 tidak selalu sama dengan species 0,1,2), kita gunakan:
- **Cross-tabulation** — melihat pemetaan antara cluster dan species.
- **Adjusted Rand Index (ARI)** — mengukur kesamaan antara dua pengelompokan, dikoreksi terhadap *chance*. Nilai ARI berkisar dari -1 hingga 1, di mana 1 berarti identik sempurna.

In [ ]:
cross_tab = pd.crosstab(df_pca['species'], df_pca['cluster'],
                        rownames=['Species'], colnames=['Cluster'])
print('Cross-Tabulation (Species vs Cluster):')
print(cross_tab)

ari = adjusted_rand_score(y_true, cluster_labels)
print(f'\nAdjusted Rand Index (ARI): {ari:.4f}')

### Interpretasi

Dari cross-tabulation dan ARI, kita bisa melihat seberapa baik K-Means mampu mereproduksi struktur alami data. ARI mendekati 1 menunjukkan bahwa cluster yang terbentuk sangat cocok dengan spesies asli. Biasanya setosa tercluster sempurna, sementara versicolor dan virginica mungkin ada sedikit tumpang tindih.

## 4. Elbow Method — Menentukan k Optimal

Elbow Method menggunakan **Within-Cluster Sum of Squares (WCSS) / inertia** untuk setiap nilai k. Nilai k optimal dipilih pada titik "siku" (elbow), yaitu di mana penurunan inertia mulai melandai.

In [ ]:
inertia_values = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca)
    inertia_values.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia_values, marker='o', linestyle='--', color='#3498db')
plt.axvline(x=3, color='red', linestyle=':', label='k=3 (tebakan awal)')
plt.xlabel('Jumlah Cluster (k)')
plt.ylabel('Inertia (WCSS)')
plt.title('Elbow Method untuk Menentukan k Optimal')
plt.xticks(k_range)
plt.legend()
plt.tight_layout()
plt.show()

### Analisis Elbow

Dari plot elbow di atas, terlihat bahwa penurunan inertia paling tajam terjadi dari k=1 ke k=2 dan k=2 ke k=3. Setelah k=3, penurunan mulai melandai. Ini mengonfirmasi bahwa **k=3** adalah pilihan yang optimal — sesuai dengan jumlah spesies asli dalam dataset.

## 5. Evaluasi dengan Silhouette Score

**Silhouette Score** mengukur seberapa mirip suatu titik dengan clusternya sendiri dibandingkan cluster lain. Rentang: -1 hingga 1.
- **~1**: Cluster sangat padat dan terpisah baik.
- **~0**: Cluster tumpang tindih.
- **<0**: Titik mungkin salah cluster.

Kita hitung silhouette score untuk k=2 hingga k=5.

In [ ]:
silhouette_scores = []
k_range_sil = range(2, 6)

for k in k_range_sil:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    score = silhouette_score(X_pca, labels)
    silhouette_scores.append(score)
    print(f'k={k}: Silhouette Score = {score:.4f}')

plt.figure(figsize=(8, 5))
plt.bar([str(k) for k in k_range_sil], silhouette_scores,
        color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'])
plt.xlabel('Jumlah Cluster (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score untuk Berbagai Nilai k')
plt.ylim(0, 1)
for i, score in enumerate(silhouette_scores):
    plt.text(i, score + 0.02, f'{score:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### Analisis Silhouette

Silhouette Score tertinggi biasanya terjadi pada k=2 atau k=3, tergantung sebaran data. Jika k=2 memberikan skor lebih tinggi, itu karena setosa sangat terpisah, sementara versicolor dan virginica masih berdekatan. Namun, dengan mempertimbangkan pengetahuan domain dan Elbow Method, **k=3** tetap menjadi pilihan yang paling informatif karena sesuai dengan jumlah spesies nyata.

## 6. Visualisasi Perbandingan: PCA (Label Asli) vs K-Means

Mari kita tampilkan side-by-side untuk perbandingan visual yang lebih jelas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot kiri: label asli
for i, species in enumerate(target_names):
    subset = df_pca[df_pca['species'] == species]
    axes[0].scatter(subset['PC1'], subset['PC2'],
                    c=colors[i], label=species.capitalize(),
                    edgecolors='k', alpha=0.7, s=50)
axes[0].set_title('Ground Truth (Species Asli)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend()

# Plot kanan: hasil clustering
for cluster_id in range(3):
    subset = df_pca[df_pca['cluster'] == cluster_id]
    axes[1].scatter(subset['PC1'], subset['PC2'],
                    c=colors[cluster_id], label=f'Cluster {cluster_id}',
                    edgecolors='k', alpha=0.7, s=50)
axes[1].scatter(centroids[:, 0], centroids[:, 1],
                c='black', marker='X', s=150,
                label='Centroids', edgecolors='white', linewidths=1.5)
axes[1].set_title('K-Means Clustering (k=3)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend()

plt.suptitle('Perbandingan: Ground Truth vs K-Means Clustering', fontsize=14)
plt.tight_layout()
plt.show()

## Kesimpulan & Insight

1. **PCA** berhasil mereduksi dimensi Iris dari 4 menjadi 2 komponen dengan tetap mempertahankan ~95%+ variansi data. Visualisasi PCA menunjukkan bahwa setosa mudah dipisahkan, sementara versicolor dan virginica memiliki overlap.

2. **K-Means Clustering (k=3)** mampu mengelompokkan data dengan cukup baik. Dilihat dari cross-tabulation dan ARI, cluster yang terbentuk sangat sesuai dengan spesies asli, terutama untuk setosa yang tercluster sempurna.

3. **Elbow Method** mengonfirmasi bahwa k=3 adalah jumlah cluster yang optimal — sesuai dengan jumlah spesies dalam dataset.

4. **Silhouette Score** memberikan evaluasi kuantitatif bahwa cluster yang terbentuk cukup padat dan terpisah.

5. **Unsupervised Learning** terbukti efektif dalam menemukan struktur intrinsik data tanpa perlu label — sebuah pendekatan yang sangat berguna ketika labeled data tidak tersedia.